# 2. Baseline Model (Single LightGBM)

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
print("Loading data...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
TARGET_COL = [col for col in train.columns if col not in test.columns and col != 'id'][0]

In [ ]:
# 2. Automatically find the Target Column 
# (It's the column that is in 'train' but missing from 'test', except 'id')
TARGET_COL = [col for col in train.columns if col not in test.columns and col != 'id'][0]
print(f"🎯 Automatically detected Target Column: '{TARGET_COL}'\n")

In [ ]:
# Save IDs for the submission file
test_ids = test['id']

In [ ]:
X = train.drop(columns=['id', TARGET_COL])
y = train[TARGET_COL]
X_test = test.drop(columns=['id'])

In [ ]:
print("Fixing Target Labels (Absence -> 0, Presence -> 1)...")
y = y.map({'Absence': 0, 'Presence': 1})

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on both to be safe
    le.fit(pd.concat([X[col], X_test[col]]).astype(str))
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [ ]:
# Cross-Validation Setup
print("\n🚀 Training The Holy Trinity (LightGBM + XGBoost + CatBoost)...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [ ]:
# Define the model
model = LGBMClassifier(
    n_estimators=800,       # Number of trees
    learning_rate=0.03,     # Slower learning rate usually gives better accuracy
    max_depth=6,            # Control tree depth to prevent overfitting
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Train the model
    model.fit(X_train, y_train)
    
    # PREDICT PROBABILITIES for ROC AUC! ([:, 1] gets the probability of class 1)
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    # Predict on the test set
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Calculate fold score
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"✅ Fold {fold+1} ROC AUC Score: {fold_auc:.5f}")
